In [0]:
CREATE OR REPLACE TABLE nyc_mobility.validation.weather_validation AS

WITH total_rows AS (
    SELECT COUNT(*) AS total_rows
    FROM nyc_mobility.clean.weather_silver
),

dq_results AS (

-- ELEVATION

SELECT
    'elevation' AS column_name,
    'Validity' AS data_quality_check,
    COUNT(*) AS failed_rows
FROM nyc_mobility.clean.weather_silver
WHERE elevation IS NULL
   OR elevation < -500
   OR elevation > 9000

UNION ALL

-- LATITUDE (NYC ACCURACY)

SELECT
    'latitude',
    'Accuracy',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE latitude IS NULL
   OR latitude NOT BETWEEN 40.4 AND 41.0

UNION ALL

-- LONGITUDE (NYC ACCURACY)

SELECT
    'longitude',
    'Accuracy',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE longitude IS NULL
   OR longitude NOT BETWEEN -74.3 AND -73.6

UNION ALL

-- OBSERVATION TIME COMPLETENESS

SELECT
    'observation_time',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE observation_time IS NULL

UNION ALL

-- OBSERVATION TIME TIMELINESS

SELECT
    'observation_time',
    'Timeliness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE observation_time > CURRENT_TIMESTAMP()

UNION ALL

-- PRECIPITATION COMPLETENESS

SELECT
    'precipitation',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE precipitation IS NULL

UNION ALL

-- PRECIPITATION ACCURACY

SELECT
    'precipitation',
    'Accuracy',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE precipitation < 0

UNION ALL

-- RAIN COMPLETENESS

SELECT
    'rain',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE rain IS NULL

UNION ALL

-- RAIN ACCURACY

SELECT
    'rain',
    'Accuracy',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE rain < 0

UNION ALL

-- SNOWFALL COMPLETENESS

SELECT
    'snowfall',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE snowfall IS NULL

UNION ALL

-- SNOWFALL ACCURACY

SELECT
    'snowfall',
    'Accuracy',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE snowfall < 0

UNION ALL

-- TEMPERATURE COMPLETENESS

SELECT
    'temperature_2m',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE temperature_2m IS NULL

UNION ALL

-- TEMPERATURE VALIDITY

SELECT
    'temperature_2m',
    'Validity',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE temperature_2m < -90
   OR temperature_2m > 60

UNION ALL

-- TIMEZONE COMPLETENESS

SELECT
    'timezone',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE timezone IS NULL

UNION ALL

-- TIMEZONE CONSISTENCY

SELECT
    'timezone',
    'Consistency',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE timezone <> 'America/New_York'

UNION ALL

-- WEATHER CODE COMPLETENESS

SELECT
    'weather_code',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE weather_code IS NULL

UNION ALL

-- WEATHER CODE VALIDITY

SELECT
    'weather_code',
    'Validity',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE weather_code NOT IN (
    0,1,2,3,
    45,48,
    51,53,55,
    56,57,
    61,63,65,
    66,67,
    71,73,75,
    77,
    80,81,82,
    85,86,
    95,96,99
)

UNION ALL

-- WIND SPEED COMPLETENESS

SELECT
    'wind_speed_10m',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE wind_speed_10m IS NULL

UNION ALL

-- WIND SPEED VALIDITY

SELECT
    'wind_speed_10m',
    'Validity',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE wind_speed_10m < 0
   OR wind_speed_10m > 500

UNION ALL

-- BUSINESS GRAIN UNIQUENESS

SELECT
    'latitude, longitude, observation_time',
    'Uniqueness',
    COUNT(*)
FROM (
    SELECT
        latitude,
        longitude,
        observation_time
    FROM nyc_mobility.clean.weather_silver
    GROUP BY
        latitude,
        longitude,
        observation_time
    HAVING COUNT(*) > 1
) d

)

SELECT
    column_name AS `Column`,
    data_quality_check AS data_quality_check,
    failed_rows AS failed_rows,
    total_rows AS total_rows,
    ROUND(
        100.0 * failed_rows / total_rows,
        2
    ) AS `Percentage`,
    CASE
        WHEN failed_rows = 0 THEN 'PASS'
        WHEN (100.0 * failed_rows / total_rows) < 5 THEN 'WARN'
        ELSE 'FAIL'
    END AS status
FROM dq_results
CROSS JOIN total_rows
ORDER BY
    `Column`,
    data_quality_check;